# TurboQuant Plus Features Demo

This notebook demonstrates all the new features from turboquant_plus that have been added to turboquant-app.

## Features Covered

1. **Turbo Format Presets** (turbo2, turbo3, turbo4)
2. **PolarQuant Algorithm**
3. **Sparse V Decoding**
4. **Asymmetric K/V Support**
5. **Outlier Channel Handling**
6. **Layer-Adaptive Mode**
7. **Norm Correction**

Let's get started!

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Import all turboquant_plus features
from core import (
    TURBO2, TURBO3, TURBO4, get_format, create_codec_from_format,
    PolarQuantConfig, PolarQuantCodec, polar_quant_roundtrip,
    SparseVDecoder, SparseKVCache,
    AsymmetricKVConfig, AsymmetricKVCache, create_asymmetric_cache,
    OutlierConfig, OutlierHandler, OutlierAwareCodec,
    LayerAdaptiveConfig, LayerAdaptiveKVCache, create_layer_adaptive_cache,
    NormCorrectionConfig, NormCorrector, NormCorrectedCodec,
)

print("✓ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Turbo Format Presets

Pre-defined compression formats: turbo2 (6.4x), turbo3 (4.6x), turbo4 (3.8x)

In [ ]:
# List all available formats
from core.turbo_formats import list_formats
print(list_formats())

In [ ]:
# Compare different turbo formats
dim = 4096
x = torch.randn(100, dim)

results = []

for format_name in ['turbo2', 'turbo3', 'turbo4']:
    codec = create_codec_from_format(format_name, dim=dim)
    encoded = codec.encode_key(x)
    decoded = codec.decode_key(encoded)
    
    mse = ((x - decoded) ** 2).mean().item()
    cosine = torch.nn.functional.cosine_similarity(
        x.view(-1, dim), decoded.view(-1, dim)
    ).mean().item()
    
    results.append({
        'format': format_name,
        'compression': codec.compression_factor,
        'mse': mse,
        'cosine': cosine
    })
    
    print(f"{format_name.upper()}: {codec.compression_factor:.1f}x compression, "
          f"MSE={mse:.6f}, Cosine={cosine:.4f}")

In [ ]:
# Visualize compression vs quality tradeoff
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Compression comparison
ax[0].bar([r['format'] for r in results], [r['compression'] for r in results])
ax[0].set_ylabel('Compression Factor (x)')
ax[0].set_title('Compression by Format')
ax[0].grid(axis='y', alpha=0.3)

# Quality comparison
ax[1].plot([r['compression'] for r in results], 
           [r['cosine'] for r in results], 'o-', linewidth=2, markersize=10)
ax[1].set_xlabel('Compression Factor')
ax[1].set_ylabel('Cosine Similarity')
ax[1].set_title('Quality vs Compression')
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. PolarQuant Algorithm

Advanced quantization using polar coordinates with WHT rotation

In [ ]:
# Test PolarQuant with different bit widths
x = torch.randn(100, 4096)

pq_results = []

for bits in [2, 3, 4]:
    x_rec, metrics = polar_quant_roundtrip(x, bits=bits, qjl_dim=64)
    pq_results.append(metrics)
    
    print(f"{bits}-bit PolarQuant:")
    print(f"  MSE: {metrics['mse']:.6f}")
    print(f"  Cosine: {metrics['cosine_similarity']:.4f}")
    print(f"  Compression: {metrics['compression_factor']:.1f}x\n")

## 3. Sparse V Decoding

Skip low-weight V positions for +22.8% speed at 32K context

In [ ]:
dim = 4096
seq_len = 1000  # Long sequence to show sparsity benefits

# Create decoder
decoder = SparseVDecoder(dim=dim, num_bits=4, threshold=1e-6)

# Encode V vectors
v = torch.randn(seq_len, dim)
encoded_v = decoder.codec.encode(v)

# Create sparse attention weights
attn_weights = torch.softmax(torch.randn(1, seq_len) * 10, dim=-1)

# Decode with sparsity
v_decoded = decoder.decode_sparse(encoded_v, attn_weights)

# Show sparsity statistics
stats = decoder.get_sparsity_stats()

print(f"Sparse V Decoding Results:")
print(f"  Sequence Length: {seq_len}")
print(f"  Sparsity: {stats['sparsity_percent']}")
print(f"  Skipped: {stats['skipped_positions']}/{stats['total_positions']}")
print(f"  Speedup: {stats['theoretical_speedup']}")

In [ ]:
# Visualize attention sparsity
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Attention weights distribution
ax[0].hist(attn_weights.numpy().flatten(), bins=50, edgecolor='black')
ax[0].axvline(x=1e-6, color='r', linestyle='--', label='Threshold (1e-6)')
ax[0].set_xlabel('Attention Weight')
ax[0].set_ylabel('Frequency')
ax[0].set_title('Attention Weight Distribution')
ax[0].legend()
ax[0].grid(alpha=0.3)

# Top-K attention
top_k = 50
top_values, top_indices = torch.topk(attn_weights[0], top_k)
ax[1].bar(range(top_k), top_values.numpy())
ax[1].set_xlabel('Position Rank')
ax[1].set_ylabel('Attention Weight')
ax[1].set_title(f'Top-{top_k} Attention Positions')
ax[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Asymmetric K/V Support

Independent formats for Keys and Values (e.g., q8_0 K + turbo4 V)

In [ ]:
# Create asymmetric cache
cache = create_asymmetric_cache(
    dim=4096,
    k_format="q8_0",    # High precision for Keys
    v_format="turbo4",  # Compressed Values
    enable_sparse_v=True
)

# Append data
seq_len = 100
k = torch.randn(seq_len, 4096)
v = torch.randn(seq_len, 4096)
cache.append(k, v)

# Get memory breakdown
memory = cache.get_memory_usage()

print(f"Asymmetric K/V Cache:")
print(f"  K Format: {cache.config.k_format}")
print(f"  V Format: {cache.config.v_format}")
print(f"  K Compression: {memory['k_memory']['factor']:.1f}x")
print(f"  V Compression: {memory['v_compression_factor']}")
print(f"  Overall: {memory['overall_compression_factor']}")

## 5. Outlier Channel Handling

Detect and handle high-variance channels separately

In [ ]:
# Create codec with outlier handling
codec = OutlierAwareCodec(
    dim=4096,
    main_bits=2,       # 2-bit for normal channels
    outlier_bits=8,    # 8-bit for outliers
    variance_threshold=10.0
)

# Create data with outliers
x = torch.randn(100, 4096)
x[:, 0:10] *= 100  # Make first 10 channels outliers

# Encode and decode
encoded = codec.encode(x)
decoded = codec.decode(encoded)

# Calculate metrics
mse = ((x - decoded) ** 2).mean().item()
cosine = torch.nn.functional.cosine_similarity(
    x.view(-1, 4096), decoded.view(-1, 4096)
).mean().item()

stats = codec.get_stats()

print(f"Outlier Handling Results:")
print(f"  Outliers Detected: {stats['avg_outliers']}")
print(f"  MSE: {mse:.6f}")
print(f"  Cosine Similarity: {cosine:.4f}")

## 6. Layer-Adaptive Mode

Keep last N layers at q8_0, compress earlier layers

In [ ]:
# Create layer-adaptive cache
cache = create_layer_adaptive_cache(
    num_layers=32,
    keep_last_n=8,
    default_format="turbo4",
    protected_format="q8_0",
    dim=4096
)

# Simulate appending to all layers
for layer_idx in range(32):
    k = torch.randn(50, 4096)
    v = torch.randn(50, 4096)
    cache.append(layer_idx, k, v)

# Get memory breakdown
memory = cache.get_memory_usage()

print(f"Layer-Adaptive Cache (32 layers):")
print(f"  Layers 0-23: turbo4")
print(f"  Layers 24-31: q8_0")
print(f"\n  Memory per layer sample:")
print(f"    Layer 0:  {memory['per_layer'][0]['k_format']}")
print(f"    Layer 16: {memory['per_layer'][16]['k_format']}")
print(f"    Layer 31: {memory['per_layer'][31]['k_format']}")
print(f"\n  Overall Compression: {memory['overall_compression_factor']:.1f}x")

In [ ]:
# Visualize layer format distribution
formats = [memory['per_layer'][i]['k_format'] for i in range(32)]
format_colors = {'turbo4': 'blue', 'q8_0': 'red'}
format_values = [0 if f == 'turbo4' else 1 for f in formats]

fig, ax = plt.subplots(1, 1, figsize=(12, 3))
ax.bar(range(32), format_values, color=[format_colors[f] for f in formats])
ax.set_xlabel('Layer Index')
ax.set_ylabel('Format')
ax.set_yticks([0, 1])
ax.set_yticklabels(['turbo4', 'q8_0'])
ax.set_title('Layer-Adaptive Format Distribution')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Norm Correction

Per-token/layer correction for improved perplexity

In [ ]:
from core.codec import TurboQuantCodec, TurboQuantConfig

# Create base codec
dim = 4096
base_codec = TurboQuantCodec(dim, TurboQuantConfig(num_bits=4))

# Wrap with norm correction
codec = NormCorrectedCodec(
    base_codec,
    NormCorrectionConfig(),
    calibrate=True
)

# Calibrate
calibration_data = [torch.randn(5, dim) for _ in range(10)]
stats = codec.calibrate(calibration_data)

print(f"Norm Correction Calibration:")
print(f"  MSE Before: {stats['mse_before']:.6f}")
print(f"  MSE After: {stats['mse_after']:.6f}")
improvement = (stats['mse_before'] - stats['mse_after']) / stats['mse_before'] * 100
print(f"  Improvement: {improvement:.1f}%")

## Summary

All turboquant_plus features are now available in turboquant-app!

In [ ]:
# Quick reference table
import pandas as pd

features = pd.DataFrame({
    'Feature': [
        'Turbo Formats',
        'PolarQuant',
        'Sparse V',
        'Asymmetric K/V',
        'Outlier Handling',
        'Layer-Adaptive',
        'Norm Correction'
    ],
    'File': [
        'core/turbo_formats.py',
        'core/polar_quant.py',
        'core/sparse_v.py',
        'core/asymmetric_kv.py',
        'core/outlier.py',
        'core/layer_adaptive.py',
        'core/norm_correction.py'
    ],
    'Benefit': [
        '6.4x compression (turbo2)',
        'WHT rotation + polar coords',
        '+22.8% decode speed',
        'Optimal K/V quality',
        'Handle high-variance channels',
        '3.5x with quality layers',
        '-1.17% perplexity on CUDA'
    ]
})

print(features.to_string(index=False))